# Nube Fit: ¿subimos el precio?
Elasticidad, precio óptimo con reglas, efecto cruzado entre segmentos y churn máximo para los segmentos Constantes y Casuales.

## 1. Datos del caso

In [1]:
import numpy as np, pandas as pd
 
D = 0.008   # d = tasa de descuento mensual (0.8%)
 
datos = pd.DataFrame({
    'segmento': ['Constantes', 'Casuales'],
    'precio':   [20, 12],      # P = precio de hoy
    'costo':    [8, 8],        # C = costo por cliente
    'unidades': [1000, 2000],  # Q = unidades al mes hoy
    'eps_x':    [0.5, 0.2],    # εx = efecto de SU precio en las unidades del OTRO segmento
    'churn':    [0.03, 0.06],  # c = churn mensual
}).set_index('segmento')
datos

,precio,costo,unidades,eps_x,churn
segmento,,,,,
Constantes,20,8,1000,0.5,0.03
Casuales,12,8,2000,0.2,0.06


## 2. Elasticidad por segmento

In [2]:
# Pruebas de precio anteriores: a cada precio, cuántas unidades se vendieron al mes
pruebas = pd.DataFrame({
    'segmento': ['Constantes'] * 5 + ['Casuales'] * 5,
    'precio':   [18, 19, 20, 21, 22,  10.8, 11.4, 12, 12.6, 13.2],
    'unidades': [1159, 1074, 1000, 934, 875,  2603, 2274, 2000, 1770, 1576],
})
pruebas

,segmento,precio,unidades
0,Constantes,18.0,1159
1,Constantes,19.0,1074
2,Constantes,20.0,1000
3,Constantes,21.0,934
4,Constantes,22.0,875
5,Casuales,10.8,2603
6,Casuales,11.4,2274
7,Casuales,12.0,2000
8,Casuales,12.6,1770
9,Casuales,13.2,1576


In [3]:
# EJEMPLO resuelto: elasticidad de Constantes
g = pruebas[pruebas.segmento == 'Constantes']
 
# Pendiente de la recta ln(unidades) contra ln(precio): es la elasticidad ε (como en el Demo 1)
eps_const = np.polyfit(np.log(g.precio), np.log(g.unidades), 1)[0]
print(f'Elasticidad de Constantes: {eps_const:.2f}')

Elasticidad de Constantes: -1.40


In [4]:
g = pruebas[pruebas.segmento == 'Casuales']

eps_casu = np.polyfit(np.log(g.precio), np.log(g.unidades), 1)[0]
print(f'Elasticidad de Casuales: {eps_casu:.2f}')

datos['eps'] = [round(eps_const, 1), round(eps_casu, 1)]
datos

Elasticidad de Casuales: -2.50


,precio,costo,unidades,eps_x,churn,eps
segmento,,,,,,
Constantes,20,8,1000,0.5,0.03,-1.4
Casuales,12,8,2000,0.2,0.06,-2.5


In [5]:
# Cuenta rápida con solo dos precios de Constantes ($20 y $22)
q20, q22 = 1000, 875
print('%ΔQ ÷ %ΔP =', round(((q22 - q20) / q20) / ((22 - 20) / 20), 2))
print('con logaritmos =', round(np.log(q22 / q20) / np.log(22 / 20), 2))

%ΔQ ÷ %ΔP = -1.25
con logaritmos = -1.4


## 3. Precio óptimo y restricciones

In [6]:
# Las mismas funciones del Demo 2 de Ritmo (no hay que cambiarlas)
def precio_optimo(C, eps):
    return C * eps / (1 + eps)                 # P* = C × ε ÷ (1 + ε)
 
def aplicar_reglas(p_hoy, p_opt, C):
    p = min(p_opt, p_hoy * 1.10)               # tope: sube máximo 10%
    p = max(p, p_hoy * 0.90)                   # tope: baja máximo 10%
    p = max(p, 1.2 * C)                        # margen mínimo: precio ≥ 1.2 × C
    return p
 
def utilidad(p, p_hoy, C, Q, eps):
    unidades = Q * (p / p_hoy) ** eps          # cuántas unidades se venden a p
    return (p - C) * unidades                  # utilidad = (p − C) × unidades
 
def clv(m, c, d=D):
    return m * (1 - c) / (c + d)               # CLV = m × (1 − c) ÷ (c + d)
 
def churn_maximo(m_nuevo, clv_hoy, d=D):
    return (m_nuevo - clv_hoy * d) / (m_nuevo + clv_hoy)   # c'

In [7]:
# EJEMPLO resuelto: segmento Constantes
s = datos.loc['Constantes']
 
p_opt  = precio_optimo(s.costo, s.eps)
p_prop = aplicar_reglas(s.precio, p_opt, s.costo)
 
print(f'Precio óptimo P*:          ${p_opt:.2f}')
print(f'Precio propuesto (reglas): ${p_prop:.2f}')
print(f'Utilidad hoy:      {utilidad(s.precio, s.precio, s.costo, s.unidades, s.eps):,.0f}')
print(f'Utilidad propuesta:{utilidad(p_prop, s.precio, s.costo, s.unidades, s.eps):,.0f}')

Precio óptimo P*:          $28.00
Precio propuesto (reglas): $22.00
Utilidad hoy:      12,000
Utilidad propuesta:12,251


In [8]:
u_opt = utilidad(p_opt, s.precio, s.costo, s.unidades, s.eps)
u_prop = utilidad(p_prop, s.precio, s.costo, s.unidades, s.eps)
print(f'Utilidad en P* (${p_opt:.2f}): {u_opt:,.0f}')
print(f'Utilidad con la regla (${p_prop:.2f}): {u_prop:,.0f}')
print(f'La restricción cuesta: {u_opt - u_prop:,.0f}')

Utilidad en P* ($28.00): 12,487
Utilidad con la regla ($22.00): 12,251
La restricción cuesta: 236


In [9]:
s = datos.loc['Casuales']

p_opt  = precio_optimo(s.costo, s.eps)
p_prop = aplicar_reglas(s.precio, p_opt, s.costo)

print(f'Precio óptimo P*:          ${p_opt:.2f}')
print(f'Precio propuesto (reglas): ${p_prop:.2f}')
print(f'Utilidad hoy:      {utilidad(s.precio, s.precio, s.costo, s.unidades, s.eps):,.0f}')
print(f'Utilidad propuesta:{utilidad(p_prop, s.precio, s.costo, s.unidades, s.eps):,.0f}')
u_opt = utilidad(p_opt, s.precio, s.costo, s.unidades, s.eps)
print(f'Utilidad en P* sin reglas: {u_opt:,.0f}')

Precio óptimo P*:          $13.33
Precio propuesto (reglas): $13.20
Utilidad hoy:      8,000
Utilidad propuesta:8,195
Utilidad en P* sin reglas: 8,197


## 4. Efecto cruzado entre segmentos

In [10]:
# La función nueva: utilidad de los DOS segmentos juntos, con elasticidad cruzada
def utilidad_total(p_const, p_casu):
    a, b = datos.loc['Constantes'], datos.loc['Casuales']
    q_const = a.unidades * (p_const / a.precio) ** a.eps * (p_casu / b.precio) ** b.eps_x
    q_casu  = b.unidades * (p_casu / b.precio) ** b.eps * (p_const / a.precio) ** a.eps_x
    u_const = (p_const - a.costo) * q_const
    u_casu  = (p_casu - b.costo) * q_casu
    return u_const, u_casu
 
# EJEMPLO resuelto: sube SOLO Constantes a $22 (Casuales se queda en $12)
hoy = utilidad_total(20, 12)
sube = utilidad_total(22, 12)
print(f'Constantes: {hoy[0]:,.0f} -> {sube[0]:,.0f}')
print(f'Casuales:   {hoy[1]:,.0f} -> {sube[1]:,.0f}')
print(f'Total:      {sum(hoy):,.0f} -> {sum(sube):,.0f}')

Constantes: 12,000 -> 12,251
Casuales:   8,000 -> 8,390
Total:      20,000 -> 20,642


In [11]:
nuevo = utilidad_total(22, 13.2)
print(f'Constantes: {hoy[0]:,.0f} -> {nuevo[0]:,.0f}')
print(f'Casuales:   {hoy[1]:,.0f} -> {nuevo[1]:,.0f}')
print(f'Total:      {sum(hoy):,.0f} -> {sum(nuevo):,.0f}')
print(f'Suma de las utilidades propuestas por separado: {12251 + 8195:,}')
print(f'Valor de contar el efecto cruzado: {sum(nuevo) - (12251 + 8195):,.0f}')

Constantes: 12,000 -> 12,487
Casuales:   8,000 -> 8,595
Total:      20,000 -> 21,082
Suma de las utilidades propuestas por separado: 20,446
Valor de contar el efecto cruzado: 636


## 5. Churn máximo

In [12]:
filas = {}
for seg, p_prop in [('Constantes', 22), ('Casuales', 13.2)]:
    s = datos.loc[seg]
    clv_hoy = clv(s.precio - s.costo, s.churn)
    m_nuevo = p_prop - s.costo
    filas[seg] = {'CLV hoy': round(clv_hoy, 1), 'margen nuevo': m_nuevo,
                  "churn máximo c'": round(churn_maximo(m_nuevo, clv_hoy), 4)}
maximos = pd.DataFrame(filas).T
maximos

,CLV hoy,margen nuevo,churn máximo c'
Constantes,306.3,14.0,0.0361
Casuales,55.3,5.2,0.0786


In [13]:
tabla = []
for e_c in (0.5, 1, 2.5):
    fila = {'ε(c)': e_c}
    for seg in ('Constantes', 'Casuales'):
        nuevo_c = datos.loc[seg, 'churn'] * (1 + e_c * 0.10)
        fila[f'Churn nuevo {seg}'] = f'{nuevo_c:.2%}'
        fila[f'¿Cabe? {seg}'] = 'Sí' if nuevo_c <= maximos.loc[seg, "churn máximo c'"] else 'No'
    tabla.append(fila)
pd.DataFrame(tabla)

,ε(c),Churn nuevo Constantes,¿Cabe? Constantes,Churn nuevo Casuales,¿Cabe? Casuales
0,0.5,3.15%,Sí,6.30%,Sí
1,1.0,3.30%,Sí,6.60%,Sí
2,2.5,3.75%,No,7.50%,Sí
